# 12D2 — Quarantined Response Resolution

This stage resolves six production responses without changing or deleting the original API envelopes. Four lexical directness flags are adjudicated against the disclosed relationship, and two overlong quotations are replaced with shorter exact contiguous excerpts. Every override and rationale is frozen in a versioned configuration file.

## 1. Load immutable inputs and resolution contract

In [1]:
from pathlib import Path
import hashlib,json,sys
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name!='sp500_critical_minerals':
    candidate=ROOT/'geopolitical_exposure'/'sp500_critical_minerals'
    if candidate.exists(): ROOT=candidate
CONFIG=ROOT/'config'; DATA=ROOT/'data'/'processed'; PROD=ROOT/'production_extraction_v2_1'; RESOLVED=ROOT/'resolved_extraction_v2_1'; RESOLVED.mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT/'src')); from production_extraction import validate_result
contract_path=CONFIG/'quarantined_response_resolution_v1.json'; contract=json.loads(contract_path.read_text())
audit=pd.read_parquet(PROD/'production_validation_audit.parquet'); original_detail=pd.read_parquet(PROD/'validated_document_exposures.parquet')
records={x['document_id']:x for x in map(json.loads,(DATA/'sp500_retrieval_records.jsonl').read_text().splitlines())}
quarantined=set(audit.loc[~audit.valid,'document_id']); configured=set(contract['documents'])
assert quarantined==configured,f'Contract mismatch: quarantined={quarantined}, configured={configured}'
display(audit.loc[~audit.valid])

,document_id,firm_id,valid,attempts,retried,issues,hashes_match,exposures
499,0001002910-24-000090,21808,False,3,True,exposure_1:possible_intermediary_marked_direct,True,1
576,0000037996-24-000144,20750,False,3,True,exposure_1:possible_intermediary_marked_direct...,True,2
670,0001002910-24-000101,21808,False,3,True,exposure_1:possible_intermediary_marked_direct,True,1
820,0000107815-24-000248,21931,False,3,True,exposure_1:possible_intermediary_marked_direct,True,1
863,0001002910-24-000111,21808,False,3,True,exposure_1:possible_intermediary_marked_direct,True,1
919,0000723125-24-000047,7065,False,3,True,exposure_1:quote_over_60_words,True,1


## 2. Apply documented resolutions to copies

Quote replacements must occur exactly in their declared source passages. Accepted validator exceptions are removed only when explicitly named in the frozen contract.

In [2]:
resolution_rows=[]; resolved_detail=[]
for doc in sorted(quarantined):
    env=json.loads((PROD/'responses'/f'{doc}.json').read_text()); result=json.loads(json.dumps(env['result'])); rule=contract['documents'][doc]; record=records[doc]
    for number,quote in rule.get('quote_replacements',{}).items(): result['exposures'][int(number)-1]['evidence_quote']=quote
    raw_issues=validate_result(result,record); remaining=sorted(set(raw_issues)-set(rule.get('accepted_issues',[])))
    assert not remaining,f'{doc} unresolved issues: {remaining}'
    for x in result['exposures']: resolved_detail.append({'document_id':doc,'firm_id':str(result['firm_id']),'document_type':result['document_type'],'available_date':result['available_date'],**x})
    resolution_rows.append({'document_id':doc,'original_issues':'|'.join(audit.set_index('document_id').loc[doc,'issues'].split('|')),'accepted_issues':'|'.join(rule.get('accepted_issues',[])),'quote_replacements':len(rule.get('quote_replacements',{})),'remaining_issues':len(remaining),'rationale':rule['rationale']})
resolution_audit=pd.DataFrame(resolution_rows); resolved_detail=pd.DataFrame(resolved_detail)
display(resolution_audit); resolution_audit.to_parquet(RESOLVED/'quarantine_resolution_audit.parquet',index=False)

,document_id,original_issues,accepted_issues,quote_replacements,remaining_issues,rationale
0,0000037996-24-000144,exposure_1:possible_intermediary_marked_direct...,exposure_1:possible_intermediary_marked_direct,1,0,Ford directly acquires and uses the named raw-...
1,0000107815-24-000248,exposure_1:possible_intermediary_marked_direct,exposure_1:possible_intermediary_marked_direct,0,0,The filing explicitly states that both the fir...
2,0000723125-24-000047,exposure_1:quote_over_60_words,,1,0,"The relationship is supported, but the evidenc..."
3,0001002910-24-000090,exposure_1:possible_intermediary_marked_direct,exposure_1:possible_intermediary_marked_direct,0,0,The exposure is Ameren Missouri's own consumpt...
4,0001002910-24-000101,exposure_1:possible_intermediary_marked_direct,exposure_1:possible_intermediary_marked_direct,0,0,Ameren Missouri directly consumes enriched ura...
5,0001002910-24-000111,exposure_1:possible_intermediary_marked_direct,exposure_1:possible_intermediary_marked_direct,0,0,Ameren Missouri directly consumes enriched ura...


## 3. Construct final validated document-level panel

In [3]:
final_detail=pd.concat([original_detail,resolved_detail],ignore_index=True).sort_values(['document_id','mechanism','direction']).reset_index(drop=True)
final_audit=audit.copy(); final_audit.loc[final_audit.document_id.isin(quarantined),'valid']=True; final_audit.loc[final_audit.document_id.isin(quarantined),'issues']='resolved_by_frozen_review_contract'
final_detail.to_parquet(RESOLVED/'final_validated_document_exposures.parquet',index=False); final_audit.to_parquet(RESOLVED/'final_document_validation_audit.parquet',index=False)
summary=pd.DataFrame([{'documents':len(final_audit),'valid_documents':int(final_audit.valid.sum()),'exposure_records':len(final_detail),'exposure_documents':final_detail.document_id.nunique(),'exposure_firms':final_detail.firm_id.nunique(),'resolved_documents':len(quarantined),'additional_api_calls':0}])
display(summary); summary.to_parquet(RESOLVED/'resolution_summary.parquet',index=False)
assert len(final_audit)==923 and final_audit.valid.all() and len(final_detail)==int(audit.exposures.sum())

,documents,valid_documents,exposure_records,exposure_documents,exposure_firms,resolved_documents,additional_api_calls
0,923,923,204,159,80,6,0


## 4. Final integrity manifest

In [4]:
manifest={'stage':'quarantined_response_resolution','contract_sha256':hashlib.sha256(contract_path.read_bytes()).hexdigest(),'source_documents':len(final_audit),'valid_documents':int(final_audit.valid.sum()),'exposure_records':len(final_detail),'resolved_documents':len(quarantined),'original_responses_modified':False,'additional_api_calls':0}
(RESOLVED/'resolution_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n'); print(json.dumps(manifest,indent=2))

{
  "stage": "quarantined_response_resolution",
  "contract_sha256": "27bffba5d6c82f3ccdef3b620450138b6c60c25c6c09cd994d9c9b0b660ebca7",
  "source_documents": 923,
  "valid_documents": 923,
  "exposure_records": 204,
  "resolved_documents": 6,
  "original_responses_modified": false,
  "additional_api_calls": 0
}
